# Fuzzy search in modern (OCR'd) English text

This notebook applies `fuzzy-search` to a different scenario: a 19th-century
English newspaper auction advertisement, digitized via OCR. Unlike the
[historical Dutch notebook](historic_dutch_resolutions.ipynb), the spelling
itself is modern/standard English -- the noise comes entirely from OCR
errors (misread characters, broken ligatures, merged/split words), which is
the most common real-world scenario for digitized newspaper archives.

We will:
1. Do basic fuzzy phrase search for keywords in an OCR'd advertisement.
2. Tune thresholds for higher-quality vs. lower-quality OCR.
3. Use a **distractor** to avoid confusing two similar English words.
4. Use the token-based searcher to find multi-word phrases efficiently.


In [1]:
from fuzzy_search import FuzzyPhraseSearcher


## 1. Basic fuzzy phrase search

Here is an OCR'd auction advertisement. Note the OCR errors: `Tobaccos` ->
`Tobaceos`, `Commercial Buildings` -> `Commerciai Buildlngs`, and a merged
character in `HALL'S`.


In [2]:
text = ("Auction of Prime Tobaceos. The Executors of the late JOHN BENNETT, "
        "Tobacco Merchant, will Sell by AUCTION, at HALL'S Sale Room, "
        "Commerciai Buildlngs, Cork, TUESDAY the 14th October.")

phrases = ["Tobacco", "AUCTION", "Commercial Buildings", "TUESDAY"]

fuzzy_searcher = FuzzyPhraseSearcher(phrase_model=phrases)

for match in fuzzy_searcher.find_matches(text):
    print(f"{match.phrase.phrase_string:<22}{match.string:<22}offset={match.offset:<4}"
          f"sim={match.levenshtein_similarity:.2f}")


Tobacco               Tobaceos              offset=17  sim=0.80
Tobacco               Tobacco               offset=67  sim=1.00
AUCTION               AUCTION               offset=98  sim=1.00
Commercial Buildings  Commerciai Buildlngs  offset=128 sim=0.90
TUESDAY               TUESDAY               offset=156 sim=1.00


All four keywords are found despite the OCR noise, including the two-word
phrase `Commercial Buildings` matching its garbled OCR form
`Commerciai Buildlngs`.

## 2. Tuning thresholds for OCR quality

The default configuration is tuned for fairly noisy OCR/HTR. For
high-quality, modern OCR (e.g. born-digital PDFs or recent scanner output),
raising the thresholds avoids spurious matches and speeds up search, since
fewer fuzzy candidates need to be considered.


In [3]:
high_quality_config = {
    "char_match_threshold": 0.8,
    "ngram_threshold": 0.7,
    "levenshtein_threshold": 0.8,
    "max_length_variance": 1,
    "ngram_size": 3,
    "skip_size": 1,
}

strict_searcher = FuzzyPhraseSearcher(config=high_quality_config, phrase_model=phrases)

for match in strict_searcher.find_matches(text):
    print(f"{match.phrase.phrase_string:<22}{match.string:<22}offset={match.offset:<4}"
          f"sim={match.levenshtein_similarity:.2f}")


Tobacco               Tobacco               offset=67  sim=1.00
AUCTION               AUCTION               offset=98  sim=1.00
Commercial Buildings  Commerciai Buildlngs  offset=128 sim=0.90
TUESDAY               TUESDAY               offset=156 sim=1.00


With stricter thresholds, the heavily OCR-damaged `Tobaceos` (similarity
0.80 to `Tobacco`) drops out, while the exact match `Tobacco`,
`Commercial Buildings`/`Commerciai Buildlngs` (similarity 0.90, still well
above the new 0.8 threshold), `AUCTION` and `TUESDAY` remain. This
illustrates the precision/recall trade-off: loosen the thresholds for
noisier source material, tighten them to cut out borderline matches on
cleaner material.

## 3. Distractors: avoiding confusion between similar English words

Suppose we are searching for the auction role `Broker`, but the source
material also frequently contains the unrelated word `Banker`. Both words
are only two characters apart, so a naive fuzzy search would confuse them.
Registering `Banker` as a **distractor** for `Broker` tells the searcher to
reject matches that are actually closer to the distractor.


In [4]:
role_phrases = [
    {"phrase": "Broker", "distractors": ["Banker"]},
]

distractor_searcher = FuzzyPhraseSearcher(config={"filter_distractors": True}, phrase_model=role_phrases)

for sentence in [
    "The Banker will inspect the cargo before the sale begins.",
    "The Brocker will inspect the cargo before the sale begins.",
]:
    matches = distractor_searcher.find_matches(sentence)
    print(sentence, "->", matches)


The Banker will inspect the cargo before the sale begins. -> []
The Brocker will inspect the cargo before the sale begins. -> [PhraseMatch(phrase: "Broker", variant: "Broker", string: "Brocker", offset: 4, ignorecase: False, levenshtein_similarity: 0.9230769230769231)]


`Banker` is correctly rejected, while the OCR-damaged `Brocker` is still
recognized as a match for `Broker`.

## 4. Token-based search for multi-word phrases

`FuzzyTokenSearcher` indexes phrases by their constituent tokens (words)
rather than as raw character sequences. This is efficient for longer texts
and for multi-word phrases, since it only needs to look near token matches
rather than scan the whole text character by character.


In [5]:
from fuzzy_search.search.token_searcher import FuzzyTokenSearcher

token_searcher = FuzzyTokenSearcher(phrase_list=["Commercial Buildings", "AUCTION", "TUESDAY"])

for match in token_searcher.find_matches(text):
    print(f"{match.phrase.phrase_string:<22}{match.string:<22}offset={match.offset:<4}"
          f"sim={match.levenshtein_similarity:.2f}")


AUCTION               AUCTION               offset=98  sim=1.00
Commercial Buildings  Commerciai Buildlngs  offset=128 sim=0.90
TUESDAY               TUESDAY               offset=156 sim=1.00


## Next steps

See the [historic Dutch text notebook](historic_dutch_resolutions.ipynb) for
phrase variants, templates, and historical-spelling examples, and the
[readthedocs Quick Start](https://fuzzy-search.readthedocs.io/) for the full
configuration reference.
